In [ ]:
from send2trash.win.legacy import prefix_and_path

s3 = boto3.client("s3")

run_id = uuid.uuid4().hex
input_key = f"{prefix}/target_set/test.jsonl"
output_key = f"{prefix}/batch-output/{run_id}.jsonl"
# Prefix, doc_id, positive
# the other needs to pass the input sentence as well
# Prefix -> to embedding (input filter), doc_id (pass to identify), positive
# Process embedding {embedding, prefix, doc_id, positive} per record
# Calculate cosine similarity per record. for record for each doc_if
# When target sentence is in record, then remove it from the list.
# count untill filling up the top k

body = s3.get_object(Bucket=bucket, Key=input_key)["Body"].read().decode("utf-8")
j = 0
out_buf = io.StringIO()
for line in body.splitlines():
    #if j == 1:
    #    break
    #j += 1
    rec = json.loads(line)
    doc_id = rec["doc_id"]
    for i, sent in enumerate(rec["case_text"]):
        #if i == 3:
         #   break
        out_buf.write(json.dumps({"docid": doc_id, "sentid": i, "inputs": sent}) + "\n")
        # print(j,i)

s3.put_object(Bucket=bucket, Key=output_key, Body=out_buf.getvalue().encode("utf-8"))
sentences_set_uri = f"s3://{bucket}/{output_key}"
print(sentences_set_uri)

In [ ]:
import json
from pathlib import Path

In [ ]:
        emb = rec["SageMakerOutput"]["embedding"]

        key = f'{rec["doc_id"]}:{i}'
        sent_embeddings_prefix[key] = {}
        sent_embeddings_prefix[key]["embedding"] = np.array(emb, dtype="float32").tolist()
        sent_embeddings_prefix[key]["textual"] = rec["prefix"]
        sent_embeddings_prefix[key]["positive"] = rec["positive"]


emb = rec["SageMakerOutput"]["embedding"]

key = f'{rec["docid"]}:{rec["sentid"]}'
sent_embeddings_suffix[key] = {}
sent_embeddings_suffix[key]["embedding"] = np.array(emb, dtype="float32").tolist()
sent_embeddings_suffix[key]["textual"] = rec["inputs"]


In [ ]:
prefix_path = Path("embedded_data/embedded-output-query/f84d458b9c8b4cff8b8db170fca64a6c.json")
suffix_path = Path("embedded_data/embedded-output/f84d458b9c8b4cff8b8db170fca64a6c.json")

with open(prefix_path) as f:
    prefix = json.load(f)

with open(suffix_path) as f:
    suffix = json.load(f)

# create this funciton
index = index(prefix)

similatiry[docid][passagekey] = calculate_cosine_similarity_between(prefix[key], suffix[key])

for dockey, value in prefix.item():
    for value[dockey][sentkey]

score_colection["1"] = {}
score_colection["5"] = {}
score_colection["10"] = {}
score_colection["20"] = {}
score_colection["50"] = {}


for docid in docids:
    for passagekey in passagekeys:
        literal_belonging_score = is_included(query_pool = prefix[docid]["textual"], element = suffix[docid][sentid]["textual"])
        if literal_belonging_score > 0.8:
            nothing
        else:
            similarity_filtered[docid][passagekey] = similatiry[docid][passagekey]
    score_collections[topk] = max_scoring(similarity_filtered[docid], topk)

 print(score_collection[topk]["prefix_textual"], score_collection[topk]["suffix_textual"], score_collection[topk]["scores"])

 calculate_recall(score_collection)
 calculate_f1_macro(score_collection)

In [ ]:
doc_to_int, int_to_doc = {}, {}
next_doc_int = 1

ids = []
vecs = []

for key, vec in sent_embeddings.items():
    doc_id, sent_id_str = key.split(":", 1)
    sent_id = int(sent_id_str)

    if doc_id not in doc_to_int:
        doc_to_int[doc_id] = next_doc_int
        int_to_doc[next_doc_int] = doc_id
        next_doc_int += 1

    did = np.int64(doc_to_int[doc_id])
    fid = (did << np.int64(32)) | np.int64(sent_id)
    ids.append(fid)
    vecs.append(vec.astype("float32"))

X = np.vstack(vecs).astype("float32")

# (optional) cosine via inner product
faiss.normalize_L2(X)

base = faiss.IndexFlatIP(X.shape[1])
index = faiss.IndexIDMap2(base)
index.add_with_ids(X, np.asarray(ids, dtype="int64"))


In [ ]:

# Evaluation

k_list = [1, 5, 10, 20]
k_max = max(k_list)

hits_at_k = {k:0 for k in k_list}
f1_gold, f1_pred = [], []

start = 0
for rec in tqdm.tqdm(test_records, desc="evaluate"):
    end = sent_offsets.pop(0)
    doc_sentence_slice = slice(start, end)
    start = end

    prefix_vec = np.array(
        prefix_predictor.predict({"inputs": [rec["prefix"]]}),
        dtype = "float32",
    )

    # Check this masking because it is not the same exactly
    mask_same_sentence = [
        sent in rec["prefix"] for sent in rec["sentences"]
    ]
    mast_same_sentence = np.array(mask_same_sentence, dtype=bool)

    # ids of candidates to keep
    keep_ids = np.where(~mast_same_sentence)[0] + doc_sentence_slice.start
    keep_vecs = all_vecs[keep_ids]

    # faiss index
    D, I = index.search(prefix_vec, k_max)

    # drop candidates with global id not in keep_ids
    valid = [i for i in I[0] if i in keep_ids][:k_max]
    if len(valid) < k_max:
        extra = [i for i in I[0] if i not in keep_ids]
        valid.extend(extra[: k_max - len(valid)])

    # Compute metrix
    for k in k_list:
        if any(all_sentences[i] == rec["positive"] for i in valid[:k]):
            hits_at_k[k] += 1

    # Binary F1
    pred1 = [1 if all_sentences[i] == rec["positive"] else 0 for i in valid]
    f1_gold.append([1] + [0]*(len(pred1) -1))
    f1_pred.append(pred1)

# Aggregate metrics
n = len(test_records)
recall = {k: hits_at_k[k] / n for k in k_list}

# macro f1
f1_scores = [
    f1_score(g, p, zero_division=0) for g, p in zip(f1_gold, f1_pred)
]
f1_macro = {k: np.mean([f1_scores[i] for i in range(n)]) for k in k_list}

print("Recall:", recall)
print("Macro F1:", f1_macro)


In [ ]:
import sys
import os

module_directory = os.path.abspath('code/')

# Add the directory to the Python path
sys.path.append(module_directory)

from inference import model_fn, transform_fn
m = model_fn("model")
body = json.dumps({"inputs": "A quick test sentence."})
out, ctype = transform_fn(m, body, "application/json", "application/json")
print(ctype, out[:80], "...")


In [ ]:

import json, math, sys, re
from typing import Any, Iterable, Tuple, Optional

# --- core checks ---
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")  # raw control chars (must be escaped in JSON text)

def _check_numbers(x: Any, path: str="$") -> Optional[Tuple[str, float]]:
    """Return (path, value) if a non-finite number is found anywhere."""
    if isinstance(x, float):
        if not math.isfinite(x):
            return (path, x)
    elif isinstance(x, (list, tuple)):
        for i, v in enumerate(x):
            hit = _check_numbers(v, f"{path}[{i}]")
            if hit: return hit
    elif isinstance(x, dict):
        for k, v in x.items():
            hit = _check_numbers(v, f"{path}.{k}")
            if hit: return hit
    return None

def _check_schema(obj: Any) -> str:
    """Return an error string or '' if OK for schema:
       {"doc_id": str, "sent_id": int, "inputs": str}"""
    if not isinstance(obj, dict):
        return "record is not a JSON object"
    for k in ("docid", "sentid", "inputs"):
        if k not in obj:
            return f"missing required key: {k}"
    if not isinstance(obj["docid"], str) or not obj["docid"]:
        return "docid must be a non-empty string"
    if not isinstance(obj["sentid"], int):
        return "sentid must be an integer"
    if not isinstance(obj["inputs"], str) or not obj["inputs"]:
        return "inputs must be a non-empty string"
    if CONTROL_CHARS_RE.search(obj["docid"]):
        return "docid contains raw control characters"
    # (inputs may legitimately contain \n, \t etc as escaped sequences in JSON; the decoder handles that)
    return ""

def validate_jsonl_lines(lines: Iterable[bytes]) -> None:
    """Validate a JSONL stream (bytes per line). Prints first problem found and exits(1); otherwise prints OK."""
    # Detect BOM on very first bytes
    first = True
    for i, raw in enumerate(lines, 1):
        if not raw:
            # skip blank lines; JSON Lines allows them but many pipelines don’t expect them
            continue
        if first:
            first = False
            if raw.startswith(b"\xef\xbb\xbf"):  # UTF-8 BOM
                print(f"Line {i}: UTF-8 BOM detected; prefer UTF-8 without BOM.", file=sys.stderr)

        # Strict UTF-8; fail fast on decoding errors
        try:
            s = raw.decode("utf-8", "strict")
        except UnicodeDecodeError as e:
            print(f"Line {i}: UTF-8 decode error: {e}", file=sys.stderr)
            sys.exit(1)

        # Must be a single JSON value per line (no trailing commas etc.)
        try:
            rec = json.loads(s)
        except json.JSONDecodeError as e:
            head = s[:120].replace("\n", "\\n")
            print(f"Line {i}: invalid JSON ({e.msg}) at pos {e.pos}. Head: {head!r}", file=sys.stderr)
            sys.exit(1)

        # Schema & numeric sanity
        err = _check_schema(rec)
        if err:
            print(f"Line {i}: {err}", file=sys.stderr)
            sys.exit(1)

        hit = _check_numbers(rec)
        if hit:
            path, val = hit
            print(f"Line {i}: non-finite number at {path}: {val!r} (JSON forbids NaN/Infinity).", file=sys.stderr)
            sys.exit(1)

    print("Input looks good: UTF-8 OK, valid JSON Lines, schema OK, no NaN/Infinity found.")

# --- usage examples ---

# 1) Local file:
# with open("your_input.jsonl", "rb") as f:
#     validate_jsonl_lines(f)

# 2) S3 object:
# import boto3
# s3 = boto3.client("s3")
# obj = s3.get_object(Bucket="your-bucket", Key="path/to/input.jsonl")
# validate_jsonl_lines(obj["Body"].iter_lines())


import boto3, json

key = f"{prefix}/batch-output/{run_id}.jsonl"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=key)
validate_jsonl_lines(obj["Body"].iter_lines())



In [1]:
%run testing_metrics.py


=== Macro metrics ===
@ 1  precision=0.3147  recall=0.0007  f1=0.0014
@ 5  precision=0.2758  recall=0.0030  f1=0.0059
@10  precision=0.2286  recall=0.0051  f1=0.0099
@20  precision=0.1820  recall=0.0082  f1=0.0156
@50  precision=0.1330  recall=0.0147  f1=0.0262

=== Examples (5 queries) ===

--- Query: R2011_France Télécom SA v European Commission:0
Document: R2011_France Télécom SA v European Commission
Query: [MASK] <TYPE_legal_and_factual id=A13 Argument_scheme=Argument from Interpretation antecedents=A1|A3|A4|A5|A6|A7|A11> In those circumstances, the General Court was correct to hold, at paragraph 323 of the judgment under appeal, that the finding of the existence of aid depended on a number of ‘circumstances unrelated to’ the special tax regime, such as the fact that the business tax was charged annually and the level of the tax rates voted each year by the authorities in the territory in which FT had establishments.</TYPE_legal_and_factual> <TYPE_prem id=A22 Argument_scheme=Argu